# Database setup and importing libraries

In [7]:

# ==========================================
# 1. Environment configuration
# ==========================================
import common
from common import *
os.environ["PATH"] = (
    r"C:\Program Files\PostgreSQL\18\bin"
    + os.pathsep
    + os.environ["PATH"]
)

os.environ["PSYCOPG_IMPL"] = "python"


# ==========================================
# 2. Project paths
# ==========================================

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "db.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"

sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(NOTEBOOKS_DIR))


# ==========================================
# 3. Common imports
# ==========================================

from db import get_engine, setup_database


# ==========================================
# 4. Database initialization
# ==========================================

engine = get_engine()

setup_database()

with engine.connect() as conn:
    version = conn.execute(text("SELECT version()")).scalar()
    print("PostgreSQL:", version.split(",")[0])



Database: cinematch
pgvector: Enabled
Schemas: Bronze, Silver, Gold
Database setup completed!
PostgreSQL: PostgreSQL 17.11 (Debian 17.11-1.pgdg12+2) on x86_64-pc-linux-gnu


# Api

In [8]:

# ==========================================
# 5. TMDB configuration
# ==========================================

load_dotenv(PROJECT_ROOT / ".env")

TMDB_TOKEN = os.getenv("TMDB_API_TOKEN")

if not TMDB_TOKEN:
    raise ValueError("TMDB API token not found!")

headers = {
    "accept": "application/json",
    "Authorization": f"Bearer {TMDB_TOKEN}"
}

print("\nProject Root:", PROJECT_ROOT)
print("Common:", sys.modules["common"].__file__)
print("Database:", sys.modules["db"].__file__)

print("\nCineMatch initialization completed!")



Project Root: c:\Users\user\Desktop\CineMatch
Common: c:\Users\user\Desktop\CineMatch\notebooks\common.py
Database: c:\Users\user\Desktop\CineMatch\db.py

CineMatch initialization completed!


In [9]:
url = "https://api.themoviedb.org/3/movie/popular"

params = {
    "language": "en-US",
    "page": 1
}

response = requests.get(
    url,
    headers=headers,
    params=params,
    timeout=30
)

response.raise_for_status()

movies_data = response.json()

print("Status Code:", response.status_code)
print("Total Results:", movies_data["total_results"])
print("Total Pages:", movies_data["total_pages"])
print("Retrieved Movies:", len(movies_data["results"]))

Status Code: 200
Total Results: 20001
Total Pages: 1001
Retrieved Movies: 20


In [10]:
print("JSON Keys:")
print(movies_data.keys())

print("\nMovie Fields:")
print(movies_data["results"][0].keys())

JSON Keys:
dict_keys(['page', 'results', 'total_pages', 'total_results'])

Movie Fields:
dict_keys(['adult', 'backdrop_path', 'genre_ids', 'id', 'title', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'release_date', 'softcore', 'video', 'vote_average', 'vote_count'])


In [11]:
print(json.dumps(
    movies_data["results"][0],
    indent=4,
    ensure_ascii=False
))

{
    "adult": false,
    "backdrop_path": "/8Tfys3mDZVp4tNoH2ktm06a0Tau.jpg",
    "genre_ids": [
        878,
        12
    ],
    "id": 687163,
    "title": "Project Hail Mary",
    "original_language": "en",
    "original_title": "Project Hail Mary",
    "overview": "Science teacher Ryland Grace wakes up on a spaceship light years from home with no recollection of who he is or how he got there. As his memory returns, he begins to uncover his mission: solve the riddle of the mysterious substance causing the sun to die out. He must call on his scientific knowledge and unorthodox ideas to save everything on Earth from extinction.",
    "popularity": 633.8322,
    "poster_path": "/iOb2fjXLbpJgyQXe46n1WtGCnaa.jpg",
    "release_date": "2026-03-15",
    "softcore": false,
    "video": false,
    "vote_average": 8.626,
    "vote_count": 8844
}


In [18]:
movies_df = pd.DataFrame(movies_data['results'])

display(movies_df.shape)

display(movies_df.columns.T)

display(movies_df.dtypes)
display(movies_df.head())

(20, 15)

Index(['adult', 'backdrop_path', 'genre_ids', 'id', 'title',
       'original_language', 'original_title', 'overview', 'popularity',
       'poster_path', 'release_date', 'softcore', 'video', 'vote_average',
       'vote_count'],
      dtype='str')

adult                   bool
backdrop_path            str
genre_ids             object
id                     int64
title                    str
original_language        str
original_title           str
overview                 str
popularity           float64
poster_path              str
release_date             str
softcore                bool
video                   bool
vote_average         float64
vote_count             int64
dtype: object

,adult,backdrop_path,genre_ids,id,title,original_language,original_title,overview,popularity,poster_path,release_date,softcore,video,vote_average,vote_count
0,False,/8Tfys3mDZVp4tNoH2ktm06a0Tau.jpg,"[878, 12]",687163,Project Hail Mary,en,Project Hail Mary,Science teacher Ryland Grace wakes up on a spa...,633.8322,/iOb2fjXLbpJgyQXe46n1WtGCnaa.jpg,2026-03-15,False,False,8.626,8844
1,False,/8R79AZjX1Bt4219Ub3CrRMsPQsW.jpg,"[27, 878, 12]",1423191,Resident Evil,en,Resident Evil,Medical courier Bryan unwittingly finds himsel...,586.7202,/gAgWJhUSQWCgtKR8170nMkYlBda.jpg,2026-09-16,False,False,7.523,2108
2,False,/6uVRPFflyJpo8Cx872olmlGTWla.jpg,"[12, 53]",1263337,Heart of the Beast,en,Heart of the Beast,"After a harrowing plane crash, Special Forces ...",558.5362,/6sYemVUn79UPKz1ZsSG6kPU4RVy.jpg,2026-09-19,False,False,7.500,707
3,False,/njbjSn2ihutJD1lNeM4jAmeDkG9.jpg,"[12, 28, 14]",1368337,The Odyssey,en,The Odyssey,"Odysseus, the legendary King of Ithaca, embark...",452.6652,/5rhTDKUhPYvpdQIijFIs5VoWsON.jpg,2026-07-15,False,False,8.000,4735
4,False,/subj7knTn4DeS6KS7qI66EQKe9i.jpg,"[878, 28, 12]",969681,Spider-Man: Brand New Day,en,Spider-Man: Brand New Day,Fighting crime full-time as Spider-Man in a wo...,344.1411,/bjiS5ipwxb9JFy3XRRN4OAilSeX.jpg,2026-07-29,False,False,8.468,6770


In [19]:
# Data quality checks
print("Missing Values:")
display(movies_df.isnull().sum())

print("Duplicate Movie IDs:", movies_df["id"].duplicated().sum())

print("Unique Movie IDs:", movies_df["id"].nunique())

Missing Values:


adult                0
backdrop_path        0
genre_ids            0
id                   0
title                0
original_language    0
original_title       0
overview             0
popularity           0
poster_path          0
release_date         0
softcore             0
video                0
vote_average         0
vote_count           0
dtype: int64

Duplicate Movie IDs: 0
Unique Movie IDs: 20
